# Amazon Beauty RQ-VAE pipeline

End-to-end run on the Amazon Product Reviews (Beauty) dataset:
1. Build embeddings (sentence-t5-base, 768d)
2. Train the RQ-VAE
3. Generate Semantic IDs

All artifacts land under `outputs/amazon_beauty_*` because `name: amazon_beauty` in the config drives the paths.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import sys
from pathlib import Path

# Absolute path of your local rq-vae/ checkout. Hardcoded because some
# Jupyter setups run with cwd=/tmp, so detecting via Path.cwd() doesn't work.
ROOT = Path("/content/drive/MyDrive/tiger/rq-vae")
assert (ROOT / "src" / "amazon_beauty_dataloader.py").exists(), f"bad ROOT: {ROOT}"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = str(ROOT / "configs" / "amazon_beauty_config.yaml")

## 1. Build embeddings

In [5]:
from src.amazon_beauty_dataloader import prepare
from src.config import load_config

cfg = load_config(CONFIG)

# Encoding all ~259k items with sentence-t5-xxl takes hours, so reuse the
# cached artifacts if a previous run already wrote them to Drive (they live
# under outputs/ relative to ROOT, which is on Drive). Flip FORCE_REENCODE
# to rebuild from scratch.
FORCE_REENCODE = False
artifacts = [cfg["data"]["embeddings_path"], cfg["data"]["stats_path"], cfg["data"]["items_csv"]]
if not FORCE_REENCODE and all(Path(p).exists() for p in artifacts):
    print("[data] reusing cached embeddings already on Drive:")
    for p in artifacts:
        print(f"  {p}")
else:
    prepare(cfg, download=True)

[data] reusing cached embeddings already on Drive:
  outputs/amazon_beauty_embeddings.npy
  outputs/amazon_beauty_stats.npz
  outputs/amazon_beauty_items.csv


## 2. Train the RQ-VAE

In [ ]:
from src.train import train

# Reinit dead codes every 1000 steps instead of the config default (250).
# ~253 steps/epoch here, so this fires roughly once every 4 epochs.
cfg["train"]["reinit_every"] = 1000

history = train(cfg)

[train] device=cuda
[train] loaded embeddings (259204, 768)
[train] model L=3  K=256  D=32
[train] k-means init done on full corpus (259204 latents)
[train] epoch   1/20000  loss=2934086344501.7266  recon=2932377224849.2256  rq=1709098860.3684  util=['0.01', '0.05', '0.05']  ppl=['1.6', '4.5', '5.7']  uniq=0.000
[train] epoch   2/20000  loss=219097.9965  recon=1.4328  rq=219096.5616  util=['0.02', '0.06', '0.08']  ppl=['2.5', '7.9', '12.8']  uniq=0.002
[train] epoch   3/20000  loss=71716.5490  recon=14130.0544  rq=57586.4943  util=['0.02', '0.24', '0.16']  ppl=['3.6', '18.5', '30.8']  uniq=0.009
[train] step 1000: reinit [250, 212, 170] dead codes per level
[train] epoch   4/20000  loss=22231.2284  recon=381.4968  rq=21849.7316  util=['1.00', '0.93', '0.98']  ppl=['206.7', '32.9', '96.0']  uniq=0.403
[train] epoch   5/20000  loss=1937.1295  recon=25.4552  rq=1911.6743  util=['1.00', '0.92', '0.96']  ppl=['228.6', '55.3', '107.7']  uniq=0.599
[train] epoch   6/20000  loss=1626.3477  rec

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
# train and eval recon span many orders of magnitude (train starts ~1e16),
# so use a log y-axis on each so the later, smaller values stay readable
axes[0].plot(epochs, [h["train_recon"] for h in history], color="tab:blue")
axes[0].set_yscale("log")
axes[0].set_title("train recon loss (log)"); axes[0].set_xlabel("epoch")
axes[1].plot(epochs, [h["recon_loss"] for h in history], color="tab:orange")
axes[1].set_yscale("log")
axes[1].set_title("eval recon loss (log)"); axes[1].set_xlabel("epoch")
for l in range(len(history[0]["utilization"])):
    axes[2].plot(epochs, [h["utilization"][l] for h in history], label=f"L{l}")
axes[2].set_title("codebook utilization"); axes[2].set_xlabel("epoch"); axes[2].legend()
axes[3].plot(epochs, [h["sid_unique_fraction"] for h in history])
axes[3].set_title("unique SID fraction"); axes[3].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 3. Generate Semantic IDs

In [6]:
from src.generate_sids import generate

ckpt = Path(cfg["output"]["checkpoints_dir"]) / "best.pt"
generate(cfg, str(ckpt))

[gen] device=cuda
[gen] loaded outputs/amazon_beauty_checkpoints/best.pt  L=3 K=256
[gen] utilization per level: ['1.000', '1.000', '1.000']
[gen] perplexity per level:  ['221.61', '211.57', '205.01']
[gen] sid unique fraction:   0.8684  collisions=34104
[gen] wrote outputs/amazon_beauty_sids.csv  (259204 rows)

[sample] 10 items and their SIDs:
     206-7-112  {'item_id': '0205616461', 'title': 'Bio-Active Anti-Aging Serum (Firming Ultra-Hydrating Serum)', 'brand': nan, 'categories': 'Beauty, Skin Care, Face, Creams & Moisturizers', 'price': nan, 'description': 'As we age, our once youthful, healthy skin succumbs to an enzymatic imbalance that wears away the cellular network, resulting in skin thinning and aging. Combining the best of nature and cosmetic biotechnology, Bio-Active products are formulated with Enzymes that gently exfoliate the skin and stimulate regeneration for a youthful glow. Benefiting from fertile orchards in the Italian countryside, Bio-active formulas are rich in

In [7]:
import json
import pandas as pd

sids = pd.read_csv(cfg["output"]["sids_csv"])
print(f"{len(sids)} items")
display(sids.head(10))

with open(cfg["output"]["metrics_json"]) as f:
    print(json.dumps(json.load(f), indent=2))

259204 items


,item_id,title,brand,categories,price,description,sid
0,0205616461,Bio-Active Anti-Aging Serum (Firming Ultra-Hyd...,NaN,"Beauty, Skin Care, Face, Creams & Moisturizers",NaN,"As we age, our once youthful, healthy skin suc...",206-7-112
1,0558925278,Eco Friendly Ecotools Quality Natural Bamboo C...,NaN,"Beauty, Tools & Accessories, Makeup Brushes & ...",NaN,Mineral Powder Brush--Apply powder or mineral ...,161-21-193
2,0733001998,Mastiha Body Lotion,NaN,"Beauty, Skin Care, Body, Moisturizers, Lotions",NaN,"From the Greek island of Chios, this Mastiha b...",136-69-6
3,0737104473,Hello Kitty Lustre Lipstick (See sellers comme...,NaN,"Beauty, Makeup, Lips, Lipstick",NaN,Limited edition Hello Kitty Lipstick featuring...,133-62-245
4,0762451459,Stephanie Johnson Mermaid Round Snap Mirror,NaN,"Beauty, Tools & Accessories, Mirrors, Makeup M...",19.98,"The mermaid is an elusive (okay, mythical) cre...",69-45-48
5,1304139212,Set of 2 MAC Lip Care - Lip Pencil - Auburn,NaN,"Beauty, Makeup, Lips, Lip Liners",NaN,"A pencil designed for shaping, lining or filli...",70-209-76
6,130414674X,Set of 2 Benefit She Laq Makeup Sealer,NaN,"Beauty, Makeup, Makeup Sets",NaN,What it is:A magical makeup sealer. What it do...,43-200-166
7,130414089X,New Benefit Waterproof Automatic Eyeliner Pen ...,NaN,"Beauty, Makeup, Eyes, Eyeliner",NaN,Length : 13.5 cm\nColor: Black\n100% Brand new...,26-16-207
8,1304196062,Max Factor Lasting Performance Foundation -Pas...,NaN,"Beauty, Makeup, Face, Foundation",NaN,Lasting Performance Liquid Make-up feels natur...,83-141-104
9,1304146537,Set of 2 Goodskin Labs Eyliplex-2 Eye Life and...,NaN,"Beauty, Skin Care, Eyes, Dark Circle Treatments",NaN,Eyliplex-2 is a dual solution that focuses on ...,239-14-197


{
  "final_utilization": [
    1.0,
    1.0,
    1.0
  ],
  "final_perplexity": [
    221.60848999023438,
    211.57093811035156,
    205.00645446777344
  ],
  "final_sid_unique_fraction": 0.8684279563586982,
  "final_sid_collisions": 34104
}
